In [ ]:

# ============================
# IMPORTS
# ============================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTENC

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import (
    StandardScaler, LabelEncoder, OrdinalEncoder, OneHotEncoder,
    PowerTransformer, QuantileTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Custom Feature Engineering
from Notebook.classification_feature_engineering import ClassificationFeatureEngineer

# MLflow
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature


# ============================
# MLFLOW SETUP
# ============================
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("EMI_Classification_Models")

print("MLflow Experiment: EMI_Classification_Models")


In [ ]:
# ============================
# LOAD & PREP DATA
# ============================
df = pd.read_csv("clean_emi_data.csv")
df.dropna(subset=["emi_eligibility"], inplace=True)

X = df.drop(["emi_eligibility", "max_monthly_emi"], axis=1)
y = df["emi_eligibility"]

print(f"Dataset shape: {df.shape}")
print("Class distribution:")
print(df["emi_eligibility"].value_counts(normalize=True))


# ============================
# HANDLE CLASS IMBALANCE – SMOTENC
# ============================
categorical_cols = [
    "gender", "marital_status", "education", "employment_type",
    "company_type", "house_type", "existing_loans", "emi_scenario"
]

cat_idx = [X.columns.get_loc(c) for c in categorical_cols]

smote = SMOTENC(categorical_features=cat_idx, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

le = LabelEncoder()
y_encoded = le.fit_transform(y_resampled)

print("Balanced class counts:", np.bincount(y_encoded))


# ============================
# TRAIN-TEST SPLIT
# ============================
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Train:", X_train.shape, "Test:", X_test.shape)


# ============================
# FEATURE ENGINEERING + SKEW HANDLING
# ============================
fe = ClassificationFeatureEngineer()
X_train_fe = fe.fit_transform(X_train)

num_cols = X_train_fe.select_dtypes(include=["int64", "float64"]).columns
skewness = X_train_fe[num_cols].skew()

low_skew = skewness[abs(skewness) <= 0.5].index.tolist()
mid_skew = skewness[(abs(skewness) > 0.5) & (abs(skewness) <= 1)].index.tolist()
high_skew = skewness[abs(skewness) > 1].index.tolist()

nominal_cols = [
    "gender", "marital_status", "employment_type",
    "company_type", "house_type", "emi_scenario"
]
ordinal_cols = ["education"]
binary_cols = ["existing_loans"]

education_order = ["High School", "Graduate", "Professional", "Post Graduate"]

# ============================
# PREPROCESSING PIPELINES
# ============================
pipe_low = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

pipe_mid = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("power", PowerTransformer()),
    ("scale", StandardScaler())
])

pipe_high = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("quantile", QuantileTransformer(output_distribution="normal")),
    ("scale", StandardScaler())
])

pipe_nominal = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

pipe_ordinal = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(categories=[education_order]))
])

pipe_binary = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(categories=[["No", "Yes"]],
                           handle_unknown="use_encoded_value",
                           unknown_value=-1))
])

# FULL TRANSFORMER
preprocessor = ColumnTransformer([
    ("low", pipe_low, low_skew),
    ("mid", pipe_mid, mid_skew),
    ("high", pipe_high, high_skew),
    ("nominal", pipe_nominal, nominal_cols),
    ("ordinal", pipe_ordinal, ordinal_cols),
    ("binary", pipe_binary, binary_cols)
], verbose_feature_names_out=False)

# FEATURE SELECTION
feature_selector = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42),
    threshold="median"
)

print("Preprocessing pipeline prepared.")


In [ ]:
# ============================
# DEFINE MODELS
# ============================
models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random_Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "Decision_Tree": DecisionTreeClassifier(random_state=42),
    "Gradient_Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

print("Models:", list(models.keys()))

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
results = {}

# ============================
# TRAIN ALL MODELS
# ============================
for model_name, model in models.items():
    print(f"\nTraining: {model_name}")

    with mlflow.start_run(run_name=model_name):

        pipe = Pipeline([
            ("feature_eng", ClassificationFeatureEngineer()),
            ("preprocess", preprocessor),
            ("select", feature_selector),
            ("model", model)
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        y_prob = pipe.predict_proba(X_test)

        # ---- Metrics ----
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

        try:
            roc = roc_auc_score(y_test, y_prob, average="weighted", multi_class="ovr")
        except:
            roc = 0.0

        # ---- Log Metrics ----
        mlflow.log_param("model", model_name)
        mlflow.log_metrics({
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1,
            "roc_auc": roc
        })

        # ---- Confusion Matrix ----
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=le.classes_, yticklabels=le.classes_)
        plt.tight_layout()
        cm_path = f"cm_{model_name}.png"
        plt.savefig(cm_path)
        mlflow.log_artifact(cm_path)
        plt.close()

        # ---- ROC Curves ----
        plt.figure(figsize=(8, 6))
        for i, cls in enumerate(le.classes_):
            fpr, tpr, _ = roc_curve((y_test == i).astype(int), y_prob[:, i])
            plt.plot(fpr, tpr, label=cls)
        plt.plot([0, 1], [0, 1], "k--")
        plt.legend()
        plt.tight_layout()

        roc_path = f"roc_{model_name}.png"
        plt.savefig(roc_path)
        mlflow.log_artifact(roc_path)
        plt.close()

        # ---- Classification Report ----
        report = classification_report(
            y_test, y_pred, target_names=le.classes_, output_dict=True
        )
        report_df = pd.DataFrame(report).transpose()
        rep_path = f"report_{model_name}.csv"
        report_df.to_csv(rep_path)
        mlflow.log_artifact(rep_path)

        # ---- Log Model ----
        signature = infer_signature(X_train, pipe.predict(X_train))
        mlflow.sklearn.log_model(pipe, "model", signature=signature)

        # ---- Store Results ----
        results[model_name] = {
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1,
            "roc_auc": roc,
            "pipeline": pipe
        }

    print(f"{model_name}: F1 = {f1:.4f}")


In [ ]:
# ============================
# MODEL COMPARISON
# ============================
comparison = pd.DataFrame({
    "Model": list(results.keys()),
    "Accuracy": [results[m]["accuracy"] for m in results],
    "Precision": [results[m]["precision"] for m in results],
    "Recall": [results[m]["recall"] for m in results],
    "F1 Score": [results[m]["f1_score"] for m in results],
    "ROC AUC": [results[m]["roc_auc"] for m in results],
}).sort_values("F1 Score", ascending=False)

print("\nMODEL COMPARISON:")
print(comparison)

comparison.to_csv("model_comparison.csv", index=False)


# ============================
# SELECT BEST MODEL
# ============================
best_model_name = comparison.iloc[0]["Model"]
best_f1 = comparison.iloc[0]["F1 Score"]
best_pipeline = results[best_model_name]["pipeline"]

print(f"\nBest Model: {best_model_name} (F1 = {best_f1:.4f})")

with mlflow.start_run(run_name=f"BEST_{best_model_name}"):

    mlflow.log_param("best_model", best_model_name)

    for metric, value in results[best_model_name].items():
        if metric != "pipeline":
            mlflow.log_metric(metric, value)

    signature = infer_signature(X_train, best_pipeline.predict(X_train))
    mlflow.sklearn.log_model(
        best_pipeline,
        artifact_path="best_model",
        signature=signature,
        registered_model_name="EMI_Classification_Best_Model"
    )

    mlflow.log_artifact("model_comparison.csv")

# Save locally
joblib.dump(best_pipeline, f"{best_model_name}_best_model.pkl")
joblib.dump(le, "label_encoder.pkl")

print("\nPipeline completed successfully.")
print(f"Best model saved: {best_model_name}_best_model.pkl")
print("Run MLflow UI:  mlflow ui --backend-store-uri mlruns")
